# Практична робота №2 — Методи одновимірної оптимізації та аналіз їх збіжності

Вкажи номер варіанта, коли клітинка попросить — решта коду підхопить твою функцію, відрізок, x0 та точність автоматично.

In [ ]:
import numpy as np
import sympy as sp
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar

In [ ]:
from variants import VARIANTS

_default = globals().get("VARIANT", 1)
_raw = input(f"Номер варіанта (1-{len(VARIANTS)}) [{_default}]: ").strip()
VARIANT = int(_raw) if _raw else _default
assert VARIANT in VARIANTS, f"Немає варіанта {VARIANT}"

## 1. Побудова моделі

З таблиці варіантів дістаємо функцію (як рядок для SymPy), відрізок локалізації, початкове наближення для методу Ньютона та точність. `sp.lambdify` перетворює символьні f, f', f'' на швидкі numpy-функції.

In [ ]:
expr_str, a0, b0, x0_newton, eps0 = VARIANTS[VARIANT]

x = sp.symbols('x', real=True)
f_sym = sp.sympify(expr_str, locals={'x': x})
df_sym = sp.diff(f_sym, x)
d2f_sym = sp.diff(df_sym, x)

f = sp.lambdify(x, f_sym, 'numpy')
df = sp.lambdify(x, df_sym, 'numpy')
d2f = sp.lambdify(x, d2f_sym, 'numpy')

print(f"Варіант {VARIANT}")
print(f"f(x) = {f_sym}")
print(f"f'(x) = {df_sym}")
print(f"f''(x) = {d2f_sym}")
print(f"Відрізок: [{a0}, {b0}], x0 (Ньютон) = {x0_newton}, eps = {eps0}")

## 2. Графік функції — перевірка унімодальності

In [ ]:
xs = np.linspace(a0, b0, 400)
ys = f(xs)

fig1, ax1 = plt.subplots(figsize=(6, 4))
ax1.plot(xs, ys)
ax1.set_xlabel('x'); ax1.set_ylabel('f(x)'); ax1.grid(alpha=.3)
ax1.set_title(f'Варіант {VARIANT}: f(x) на [{a0}, {b0}]')
fig1.tight_layout()
fig1.savefig('function_plot.png', dpi=150)
fig1

## 3. Аналітичний розв'язок

Розв'язуємо f'(x) = 0 чисельно (`sp.nsolve`, бо для трансцендентних функцій закритої форми нема) з початковим наближенням x0, і перевіряємо достатню умову мінімуму — знак f''(x*).

In [ ]:
x_exact = float(sp.nsolve(df_sym, x, x0_newton))
f2_at_exact = float(d2f_sym.subs(x, x_exact))

print(f"Розв'язок рівняння f'(x)=0: x* = {x_exact:.6f}")
print(f"f(x*) = {f(x_exact):.6f}")
print(f"f''(x*) = {f2_at_exact:.6f} -> {'МІНІМУМ (f\'\'>0)' if f2_at_exact > 0 else 'НЕ мінімум (f\'\'<=0)'}")

## 4. Реалізація методів

Кожна функція повертає `(x*, кількість_обчислень_f, кількість_ітерацій, історія)`, де історія — список словників по кожній ітерації (для таблиць у звіті).

In [ ]:
def dichotomy(f, a, b, eps, delta=None):
    if delta is None:
        delta = eps / 10
    history = []
    n = 0
    while (b - a) / 2 > eps:
        m = (a + b) / 2
        x1, x2 = m - delta, m + delta
        f1, f2 = f(x1), f(x2)
        history.append(dict(iter=n + 1, a=a, b=b, x1=x1, x2=x2, f1=f1, f2=f2, length=b - a))
        if f1 <= f2:
            b = x2
        else:
            a = x1
        n += 1
    return (a + b) / 2, 2 * n, n, history


def golden(f, a, b, eps):
    phi = (np.sqrt(5) - 1) / 2
    x1, x2 = b - phi * (b - a), a + phi * (b - a)
    f1, f2 = f(x1), f(x2)
    n = 2
    history = []
    while (b - a) / 2 > eps:
        history.append(dict(iter=n - 1, a=a, b=b, x1=x1, x2=x2, f1=f1, f2=f2, length=b - a))
        if f1 <= f2:
            b, x2, f2 = x2, x1, f1
            x1 = b - phi * (b - a); f1 = f(x1)
        else:
            a, x1, f1 = x1, x2, f2
            x2 = a + phi * (b - a); f2 = f(x2)
        n += 1
    return (a + b) / 2, n, n - 1, history


def fibonacci(f, a, b, eps):
    fib = [1, 1]
    while fib[-1] < (b - a) / eps:
        fib.append(fib[-1] + fib[-2])
    N = len(fib) - 1
    x1 = a + fib[N - 2] / fib[N] * (b - a)
    x2 = a + fib[N - 1] / fib[N] * (b - a)
    f1, f2 = f(x1), f(x2)
    history = []
    for k in range(1, N - 1):
        history.append(dict(iter=k, a=a, b=b, x1=x1, x2=x2, f1=f1, f2=f2, length=b - a))
        if f1 <= f2:
            b, x2, f2 = x2, x1, f1
            x1 = a + fib[N - k - 2] / fib[N - k] * (b - a); f1 = f(x1)
        else:
            a, x1, f1 = x1, x2, f2
            x2 = a + fib[N - k - 1] / fib[N - k] * (b - a); f2 = f(x2)
    return (a + b) / 2, N, N - 1, history


def newton(df, d2f, x0, eps, itmax=100):
    x_k = x0
    history = []
    i = 0
    for i in range(itmax):
        fp, fpp = df(x_k), d2f(x_k)
        step = fp / fpp
        x_next = x_k - step
        history.append(dict(iter=i + 1, x=x_k, fpx=fp, fppx=fpp, step=step, x_next=x_next))
        x_k = x_next
        if abs(step) < eps:
            break
    return x_k, i + 1, i + 1, history

## 5. Запуск методів і таблиці перших ітерацій

In [ ]:
x_dich, calls_dich, it_dich, hist_dich = dichotomy(f, a0, b0, eps0)
x_gold, calls_gold, it_gold, hist_gold = golden(f, a0, b0, eps0)
x_fib, calls_fib, it_fib, hist_fib = fibonacci(f, a0, b0, eps0)
x_newt, calls_newt, it_newt, hist_newt = newton(df, d2f, x0_newton, eps0)

print("--- Дихотомія (перші рядки) ---")
print(pd.DataFrame(hist_dich).head(7).to_string(index=False, float_format=lambda v: f"{v:.6f}"))
print("\n--- Золотий переріз (перші рядки) ---")
print(pd.DataFrame(hist_gold).head(7).to_string(index=False, float_format=lambda v: f"{v:.6f}"))
print("\n--- Фібоначчі (перші рядки) ---")
print(pd.DataFrame(hist_fib).head(7).to_string(index=False, float_format=lambda v: f"{v:.6f}"))
print("\n--- Ньютона (перші рядки) ---")
print(pd.DataFrame(hist_newt).head(7).to_string(index=False, float_format=lambda v: f"{v:.6f}"))

## 6. Порівняння з SciPy (метод Брента)

`method='bounded'` шукає мінімум прямо в межах `[a0, b0]`, без потреби у валідній трибрекетній точці — надійніше для довільної функції варіанта.

In [ ]:
res = minimize_scalar(f, bounds=(a0, b0), method='bounded',
                       options={'xatol': eps0})
print(f"SciPy (bounded/Brent): x* = {res.x:.6f}, f(x*) = {res.fun:.6f}, обчислень f = {res.nfev}, ітерацій = {res.nit}")

## 7. Зведена порівняльна таблиця

In [ ]:
summary = pd.DataFrame([
    dict(method="Дихотомія", x_star=x_dich, f_star=f(x_dich), error=abs(x_dich - x_exact), calls=calls_dich, iters=it_dich),
    dict(method="Золотий переріз", x_star=x_gold, f_star=f(x_gold), error=abs(x_gold - x_exact), calls=calls_gold, iters=it_gold),
    dict(method="Фібоначчі", x_star=x_fib, f_star=f(x_fib), error=abs(x_fib - x_exact), calls=calls_fib, iters=it_fib),
    dict(method="Ньютона", x_star=x_newt, f_star=f(x_newt), error=abs(x_newt - x_exact), calls=calls_newt, iters=it_newt),
    dict(method="SciPy (Brent)", x_star=res.x, f_star=res.fun, error=abs(res.x - x_exact), calls=res.nfev, iters=res.nit),
])
print(summary.to_string(index=False, float_format=lambda v: f"{v:.6f}"))

## 8. Графік збіжності (довжина інтервалу, лог. шкала)

In [ ]:
fig2, ax2 = plt.subplots(figsize=(6, 4))
for hist, label in [(hist_dich, 'Дихотомія'), (hist_gold, 'Золотий переріз'), (hist_fib, 'Фібоначчі')]:
    lengths = [h['length'] for h in hist]
    ax2.semilogy(range(1, len(lengths) + 1), lengths, marker='o', ms=3, label=label)
ax2.set_xlabel('номер ітерації'); ax2.set_ylabel('довжина інтервалу (лог. шкала)')
ax2.grid(alpha=.3, which='both'); ax2.legend()
ax2.set_title(f'Варіант {VARIANT}: збіжність методів скорочення інтервалу')
fig2.tight_layout()
fig2.savefig('convergence_plot.png', dpi=150)
fig2

## 9. Вплив точності ε на кількість обчислень

In [ ]:
eps_list = [1e-3, 1e-5, 1e-7]
rows = []
for e in eps_list:
    _, c_d, i_d, _ = dichotomy(f, a0, b0, e)
    _, c_g, i_g, _ = golden(f, a0, b0, e)
    _, c_f, i_f, _ = fibonacci(f, a0, b0, e)
    _, c_n, i_n, _ = newton(df, d2f, x0_newton, e)
    rows.append(dict(eps=e, calls_dich=c_d, calls_golden=c_g, calls_fib=c_f, calls_newton=c_n))
eps_table = pd.DataFrame(rows)
print(eps_table.to_string(index=False))

## 10. Стійкість методу Ньютона

Запускаємо метод Ньютона з трьох різних початкових наближень (ближче до лівої межі, задане x0, ближче до правої межі) і дивимось, чи всі вони збігаються до того самого мінімуму.

In [ ]:
def newton_safe(df, d2f, x0, eps, itmax=100, blow_up=1e6):
    x_k = x0
    try:
        for i in range(itmax):
            fp, fpp = df(x_k), d2f(x_k)
            if fpp == 0:
                return x_k, i + 1, "ділення на нуль (f''=0)"
            step = fp / fpp
            x_next = x_k - step
            if not np.isfinite(x_next) or abs(x_next) > blow_up:
                return x_next, i + 1, "розбіжність (|x| -> дуже велике)"
            x_k = x_next
            if abs(step) < eps:
                return x_k, i + 1, "збіжність"
        return x_k, itmax, "не збігся за itmax ітерацій"
    except (ValueError, OverflowError, ZeroDivisionError) as exc:
        return None, None, f"помилка обчислення: {exc}"


starts = sorted({round(a0 + 0.1 * (b0 - a0), 4), x0_newton, round(b0 - 0.1 * (b0 - a0), 4)})
print(f"Досліджувані початкові наближення: {starts}\n")
for s in starts:
    xr, nit, status = newton_safe(df, d2f, s, eps0)
    xr_str = f"{xr:.6f}" if isinstance(xr, (int, float)) and xr is not None else str(xr)
    print(f"x0 = {s:>10}  ->  x* = {xr_str:>14}   ітерацій: {nit}   статус: {status}")